# Differentiable CO2 thermodynamics in JAX

Most engineering work on CO2 systems eventually wants two things from a property layer: reference accuracy near the critical point, and gradients that compose with the rest of the program. If you build sCO2 cycle models and have reached for a neural-network surrogate plus a genetic algorithm because the simulator was not differentiable, train physics-informed neural networks for geological CO2 storage, run dynamic simulation or model-predictive control on a transcritical heat pump near the critical point, or simply wrap CoolProp behind a `jax.pure_callback` and watch your `vmap`, `jit`, and GPU semantics evaporate, you have hit the wall this notebook is about. Today the choice is a trilemma. CoolProp gives Span-Wagner accuracy but no JAX autodiff. A cubic equation of state reimplemented in JAX gives autodiff but loses accuracy near the critical point, where most of the interesting CO2 engineering happens. teqp gives autodiff but lives in C++, outside the JAX ecosystem. `co2-eos` resolves the trilemma for CO2 specifically: Span-Wagner reference accuracy, native JAX semantics, `vmap` and `grad` out of the box. The four sections below take that claim in turn: accuracy against CoolProp across the regions where it matters, throughput with `vmap`, a one-line `jax.grad` through the equation of state, and an end-to-end gradient-based optimization that finds the temperature maximizing `cp` on an isobar through the Widom line.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import co2_eos as co2

# Publication-style matplotlib defaults: clean spines, no chartjunk gridlines,
# perceptually uniform colormap, modest figure size suitable for a README.
plt.rcParams.update({
    "figure.figsize": (7.0, 4.5),
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "regular",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": False,
    "legend.fontsize": 10,
    "lines.linewidth": 1.6,
    "image.cmap": "viridis",
    "font.family": "DejaVu Sans",
})

## 1. Accuracy against CoolProp

CoolProp's Span-Wagner implementation is the de facto reference for CO2 properties in scientific computing, so the first thing to establish is that `co2-eos` returns the same numbers in the regions where reproduction matters most: the supercritical bulk where most CO2 power-cycle and CCS work lives, the dense liquid where transport-property formulations are stiff, and the band around the Widom line where derivatives diverge and a cubic equation of state begins to disagree with experiment.

The figure below sweeps temperature across each of those regions and overlays `co2-eos` against `CoolProp.PropsSI` for density, isobaric heat capacity, and speed of sound. If the two libraries agree to roughly machine precision in the bulk and to within a few parts in ten to the eighth in the regions where physics amplifies floating-point noise, the cubic-EOS shortcut is closed for any CO2 problem where the answer needs to match CoolProp.

In [ ]:
# TODO: Section 1 content. Compute density, cp, and speed of sound from
# co2_eos.state_from_PT and from CoolProp.PropsSI on three temperature sweeps:
# (a) supercritical bulk along an isobar well above the critical pressure,
# (b) dense liquid along an isotherm below the critical temperature,
# (c) a sweep through the Widom line at 8 MPa.
# Plot the three properties as 3-row x 3-column small multiples, with absolute
# values in row 1-2 and relative error |co2 - cp| / |cp| on a log scale in row 3.

## 2. Throughput with vmap

The second wall a JAX user hits with CoolProp is the Python loop. A typical workflow scans `PropsSI` over thousands of states inside an inner loop, paying the Python-to-C++ boundary cost on every call. `co2-eos` evaluates a state in roughly two microseconds after JIT compilation and composes naturally with `jax.vmap`, which collapses the loop into a single fused kernel that runs once.

The benchmark below sweeps roughly ten thousand (P, T) points twice: once with `CoolProp.PropsSI` in a Python `for` loop, once with `jax.vmap(co2_eos.state_from_PT)`. Both compute the same scalar property at the same conditions. The numbers to watch are wall-clock time per point and how the gap scales as the batch grows, since the JAX kernel amortizes its compile cost across the whole batch while the CoolProp loop pays per-call overhead on every iteration.

In [ ]:
# TODO: Section 2 content. Build a (P, T) grid of ~1e4 points covering the
# supercritical region. Time three implementations on the same inputs:
# (a) CoolProp.PropsSI inside a Python for loop,
# (b) jax.vmap(co2.state_from_PT) once compiled,
# (c) the same vmap call after warm-up, scaled across batch sizes 10, 100,
#     1_000, 10_000 to show the per-point cost flattening.
# Plot per-point time vs batch size on a log-log axis, and report the
# CoolProp-to-co2_eos throughput ratio at the largest batch size in the title.

## 3. A gradient through the equation of state

This is the headline. Picking a single isobar at 8 MPa and walking temperature through the Widom line, `∂ρ/∂T` at constant pressure has a sharp negative peak where the supercritical fluid passes through its quasi-phase transition. Computing that derivative with a cubic EOS gives the wrong shape; computing it with finite differences over CoolProp gives a noisy answer that depends on the step size and the Python-boundary jitter. With `co2-eos` it is one line, written exactly as a JAX user would expect:

```python
drho_dT = jax.grad(lambda T: co2.state_from_PT(P=8e6, T=T)["density"])
```

The same call composes with `jax.vmap` to evaluate the derivative at a thousand temperatures in one fused kernel. Higher derivatives, mixed partials, gradients with respect to pressure, gradients with respect to both at once: all of them follow from the same primitive without writing any analytical derivatives by hand. The plot below shows `∂ρ/∂T` along the 8 MPa isobar and marks the Widom maximum where the derivative peaks.

In [ ]:
# TODO: Section 3 content. Define drho_dT as one line of jax.grad over
# co2.state_from_PT at P=8 MPa. Vmap it across T in [280, 340] K with a
# few hundred points. Plot rho(T) and drho/dT(T) on a shared x-axis with
# twin y-axes, mark the Widom-line temperature (where |drho/dT| peaks),
# and annotate the figure with the one-line jax.grad expression itself.

## 4. End-to-end optimization through the EOS

The final section is what the previous three are for. A common task in cycle and heat-exchanger design is to find the point of maximum `cp` along an isobar, since that is roughly where the Widom line sits and where heat-transfer effectiveness is largest. With a differentiable EOS this becomes a one-variable optimization over a smooth scalar objective: the temperature that maximizes `cp = ∂h/∂T` at fixed pressure.

Below, a gradient-based JAX optimizer ascends `cp(T)` on the 8 MPa isobar by composing `jax.grad` with `co2.state_from_PT`. The whole loop, including the property evaluation, the gradient, the update step, and the trace, fits in roughly thirty lines and stays inside JAX end to end. There is no callback to an external library, no finite-difference gradient, no warm-start guesswork. The same pattern composes with `jax.vmap` to run a batch of optimizations at different pressures in parallel, and with `jax.jit` to fuse the inner loop into a single compiled function.

In [ ]:
# TODO: Section 4 content. Define cp(T) at P=8 MPa via co2.state_from_PT,
# then grad_cp = jax.grad(cp). Run a small gradient-ascent loop (Adam or
# plain gradient ascent with a line-search style step) starting from a
# poor initial guess like T0=290 K. Record the trace of T_k and cp(T_k).
# Plot cp(T) over the whole isobar and overlay the optimization trajectory
# as a sequence of points climbing toward the Widom maximum. Print the
# converged T*, cp(T*), and iteration count.